In [1]:
import sqlite3 
conn = sqlite3.connect("telecom.db") 
cur = conn.cursor() 
cur.execute(""" 
CREATE TABLE IF NOT EXISTS customers ( 
customer_id INTEGER PRIMARY KEY, 
name TEXT, plan_type TEXT, region TEXT, join_date TEXT 
); 
""") 

In [2]:
cur.executemany(""" 
INSERT OR REPLACE INTO customers (customer_id, name, plan_type, region, join_date) 
VALUES (?,?,?,?,?) 
""", [ 
(1001,"Asha Mehta","Prepaid","Delhi","2023-05-12"), 
(1002,"Ravi Kumar","Postpaid","Mumbai","2022-12-20"), 
(1003,"Sneha Rao","Prepaid","Chennai","2023-01-18"), 
(1004,"Manoj Singh","Postpaid","Delhi","2021-11-05"), 
(1005,"Divya Jain","Prepaid","Kolkata","2023-03-28"), 
]) 
conn.commit(); conn.close() 
print("✅ telecom.db ready.") 

✅ telecom.db ready.


.2 Create a complaints.csv 
i

In [3]:
import pandas as pd  

complaints = pd.DataFrame([  
    {"complaint_id":"CMP-001","customer_id":1002,"category":"Billing","description":"Charged extra for data usage","created_at":"2025/09/25 10:45","status":"Open"},  
    {"complaint_id":"CMP-002","customer_id":1004,"category":"Network","description":"Frequent call drops in Delhi","created_at":"2025-09-25 09:30","status":"Open"},  
    {"complaint_id":"CMP-003","customer_id":1005,"category":"Recharge","description":"Recharge failed; amount deducted","created_at":"25-09-2025 14:00","status":"Closed"},  
    {"complaint_id":"CMP-004","customer_id":1002,"category":"Network","description":"Slow 4G speed at night","created_at":"2025-09-26 20:40","status":"Open"},  
    {"complaint_id":"CMP-005","customer_id":1003,"category":"Support","description":"No response to complaint","created_at":"2025-09-26 11:10","status":"Open"}  
])  

complaints.to_csv("complaints.csv", index=False)  
print("✅ complaints.csv saved.")  

✅ complaints.csv saved.


3) Build the ETL (Extract → Transform → Load) 
Create a new notebook (or new section) called etl_pipeline.

In [ ]:
import pandas as pd 
from sqlalchemy import create_engine



# 3.2 EXTRACT (pull data from DB + CSV) 

In [6]:
engine = create_engine("sqlite:///telecom.db") 
customers = pd.read_sql("SELECT * FROM customers", engine) 
# Complaints from CSV 
complaints = pd.read_csv("complaints.csv") 
print("Rows -> customers:", len(customers), " complaints:", len(complaints)) 
customers.head(), complaints.head() 

Rows -> customers: 5  complaints: 5


(   customer_id         name plan_type   region   join_date
 0         1001   Asha Mehta   Prepaid    Delhi  2023-05-12
 1         1002   Ravi Kumar  Postpaid   Mumbai  2022-12-20
 2         1003    Sneha Rao   Prepaid  Chennai  2023-01-18
 3         1004  Manoj Singh  Postpaid    Delhi  2021-11-05
 4         1005   Divya Jain   Prepaid  Kolkata  2023-03-28,
   complaint_id  customer_id  category                       description  \
 0      CMP-001         1002   Billing      Charged extra for data usage   
 1      CMP-002         1004   Network      Frequent call drops in Delhi   
 2      CMP-003         1005  Recharge  Recharge failed; amount deducted   
 3      CMP-004         1002   Network            Slow 4G speed at night   
 4      CMP-005         1003   Support          No response to complaint   
 
          created_at  status  
 0  2025/09/25 10:45    Open  
 1  2025-09-25 09:30    Open  
 2  25-09-2025 14:00  Closed  
 3  2025-09-26 20:40    Open  
 4  2025-09-26 11:10    Op

3.3 TRANSFORM (clean, standardize, and fix data) 
We correct bad/missing values instead of dropping rows.

In [8]:
# We correct bad/missing values instead of dropping rows. 
# --- Standardize text --- 
customers['region']  = customers['region'].str.title().str.strip() 
complaints['status'] = complaints['status'].str.title().str.strip() 
complaints['category'] = complaints['category'].str.title().str.strip() 
# --- Parse and standardize dates (multiple formats handled) --- 
customers['join_date']  = pd.to_datetime(customers['join_date'],  errors='coerce') 
complaints['created_at'] = pd.to_datetime(complaints['created_at'], errors='coerce') 
# Fill any unparseable dates with a sensible default (keeps rows intact) 
default_dt = pd.Timestamp('2025-09-25 00:00') 
customers['join_date']  = customers['join_date'].fillna(default_dt) 
complaints['created_at'] = complaints['created_at'].fillna(default_dt) 
# --- Fix missing IDs or text (rare in this sample, but robust) --- 
# If a complaint is missing customer_id, set to -1 and keep it for audit 
complaints['customer_id'] = complaints['customer_id'].fillna(-1).astype(int) 
complaints['description'] = complaints['description'].fillna("No description provided") 
# --- Merge: keep ALL customers (left join), attach complaints where present --- 
merged = customers.merge(complaints, on='customer_id', how='left') 
# --- Post-merge fixups for missing complaint fields (no deletion) --- 
merged['complaint_id'] = merged['complaint_id'].fillna("NO-COMPLAINT") 
merged['category']     = merged['category'].fillna("No Complaint") 
merged['status']       = merged['status'].fillna("Resolved") 
merged['created_at']   = merged['created_at'].fillna(default_dt) 
# Derived flag for analytics 
merged['is_open'] = (merged['status'] == 'Open')

Why this is robust: 
• Dates in different formats are parsed; failed parses get a default date (not 
dropped). 
• Missing complaint fields are filled with clear placeholders. 
• All customers remain in the final dataset (left join). 

3.4 LOAD (save the integrated dataset) 

In [9]:

merged.to_csv("etl_output.csv", index=False) 
print(f"✅ Pipeline complete! Created dataset with {len(merged)} rows and {merged.shape[1]} columns.") 

✅ Pipeline complete! Created dataset with 6 rows and 11 columns.


4) Validate & sanity-check 

In [10]:
# 4) Validate & sanity-check 
print(merged.head()) 
print("\nComplaints per customer:") 
print(merged.groupby(['customer_id','name']).complaint_id.count().reset_index(name='complaint_count')) 
print("\nOpen vs Closed (including no-complaint as 'Resolved'):") 
print(merged['status'].value_counts()) 
print("\nComplaints by category and region:") 
print(merged.groupby(['region','category']).complaint_id.count().reset_index(name='count')) 

   customer_id         name plan_type   region  join_date  complaint_id  \
0         1001   Asha Mehta   Prepaid    Delhi 2023-05-12  NO-COMPLAINT   
1         1002   Ravi Kumar  Postpaid   Mumbai 2022-12-20       CMP-001   
2         1002   Ravi Kumar  Postpaid   Mumbai 2022-12-20       CMP-004   
3         1003    Sneha Rao   Prepaid  Chennai 2023-01-18       CMP-005   
4         1004  Manoj Singh  Postpaid    Delhi 2021-11-05       CMP-002   

       category                   description          created_at    status  \
0  No Complaint                           NaN 2025-09-25 00:00:00  Resolved   
1       Billing  Charged extra for data usage 2025-09-25 10:45:00      Open   
2       Network        Slow 4G speed at night 2025-09-25 00:00:00      Open   
3       Support      No response to complaint 2025-09-25 00:00:00      Open   
4       Network  Frequent call drops in Delhi 2025-09-25 00:00:00      Open   

   is_open  
0    False  
1     True  
2     True  
3     True  
4     Tru

print(merged.groupby(['region','category']).complaint_id.count().reset_index(name='count')) 
Success criteria 
• All 5 customers appear (one row per customer per complaint; some customers 
may repeat if they have multiple complaints). 
• Customers with no complaints show NO-COMPLAINT / No Complaint / 
Resolved. 
• Dates look consistent (YYYY-MM-DD hh:mm). 
• No rows were deleted during cleaning. 

5) Deliverables (what students submit) 
• telecom.db 
• complaints.csv 
• etl_output.csv 
• Notebook: etl_pipeline.ipynb 
• A 3–5 line insight note 